# Image Dehazing: AOD-Net & DCPDN Results Report

This notebook evaluates and compares **AOD-Net** and **DCPDN** on two benchmark datasets:
- **RESIDE-SOTS** (synthetic outdoor hazy images)
- **O-HAZE** (real-world hazy images)

**Dark Channel Prior (DCP)** is included as a classical baseline for reference.

Metrics: **PSNR** (dB) and **SSIM**.

In [1]:
import os
import sys
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict

# Project imports
from config import (RESIDE_SOTS_HAZY, RESIDE_SOTS_GT, OHAZE_HAZY, OHAZE_GT,
                    CHECKPOINTS_DIR, OUTPUTS_DIR, DCP_CONFIG)
from datasets import RESIDEOutdoorTestDataset, OHAZEDataset
from metrics import compute_psnr, compute_ssim
from methods.dcp import DarkChannelPrior
from methods.aodnet import AODNet
from methods.dcpdn import DCPDN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 1. Load Models

In [2]:
# Load AOD-Net
aodnet = AODNet().to(device)
aodnet_ckpt = os.path.join(CHECKPOINTS_DIR, "aodnet_best.pth")
if os.path.exists(aodnet_ckpt):
    aodnet.load_state_dict(torch.load(aodnet_ckpt, map_location=device, weights_only=True))
    print(f"AOD-Net loaded from {aodnet_ckpt}")
else:
    print(f"WARNING: AOD-Net checkpoint not found at {aodnet_ckpt}")
aodnet.eval()

# Load DCPDN
dcpdn = DCPDN().to(device)
dcpdn_ckpt = os.path.join(CHECKPOINTS_DIR, "dcpdn_best.pth")
if os.path.exists(dcpdn_ckpt):
    dcpdn.load_state_dict(torch.load(dcpdn_ckpt, map_location=device, weights_only=True))
    print(f"DCPDN loaded from {dcpdn_ckpt}")
else:
    print(f"WARNING: DCPDN checkpoint not found at {dcpdn_ckpt}")
dcpdn.eval()

# DCP (no checkpoint needed)
dcp = DarkChannelPrior(**DCP_CONFIG)
print("DCP initialized")

AOD-Net loaded from c:\Users\thoma\OneDrive - Universitetet i Agder\Computer vision\Project\checkpoints\aodnet_best.pth
DCPDN loaded from c:\Users\thoma\OneDrive - Universitetet i Agder\Computer vision\Project\checkpoints\dcpdn_best.pth
DCP initialized


## 2. Load Datasets

In [3]:
IMAGE_SIZE = 256
BATCH_SIZE = 4

# RESIDE-SOTS
reside_ds = RESIDEOutdoorTestDataset(RESIDE_SOTS_HAZY, RESIDE_SOTS_GT, IMAGE_SIZE)
reside_loader = DataLoader(reside_ds, batch_size=BATCH_SIZE, num_workers=0)
print(f"RESIDE-SOTS: {len(reside_ds)} image pairs")

# O-HAZE
ohaze_ds = OHAZEDataset(OHAZE_HAZY, OHAZE_GT, IMAGE_SIZE)
ohaze_loader = DataLoader(ohaze_ds, batch_size=BATCH_SIZE, num_workers=0)
print(f"O-HAZE: {len(ohaze_ds)} image pairs")

RESIDE-SOTS: 500 image pairs
O-HAZE: 45 image pairs


## 3. Evaluation Functions

In [4]:
def evaluate_dcp_on_loader(dcp_model, dataloader, desc="DCP"):
    """Evaluate DCP on a dataloader, return per-image PSNR/SSIM and sample results."""
    psnr_list, ssim_list = [], []
    samples = []  # (hazy, dehazed, gt, name)

    for hazy, gt, names in tqdm(dataloader, desc=desc):
        for i in range(hazy.shape[0]):
            hazy_np = hazy[i].numpy().transpose(1, 2, 0)
            gt_np = gt[i].numpy().transpose(1, 2, 0)
            dehazed = dcp_model.dehaze(hazy_np)

            p = compute_psnr(dehazed, gt_np)
            s = compute_ssim(dehazed, gt_np)
            psnr_list.append(p)
            ssim_list.append(s)

            if len(samples) < 8:
                fname = names[i] if isinstance(names, (list, tuple)) else names
                samples.append((hazy_np, dehazed, gt_np, fname))

    return np.array(psnr_list), np.array(ssim_list), samples


def evaluate_model_on_loader(model, dataloader, model_name, device, desc="Model"):
    """Evaluate a deep learning model on a dataloader."""
    model.eval()
    psnr_list, ssim_list = [], []
    samples = []

    with torch.no_grad():
        for hazy, gt, names in tqdm(dataloader, desc=desc):
            hazy_dev = hazy.to(device)

            if model_name == "DCPDN":
                output, t_map, A_map = model(hazy_dev)
            else:
                output = model(hazy_dev)

            output = output.cpu()
            for i in range(output.shape[0]):
                out_np = output[i].numpy().transpose(1, 2, 0)
                gt_np = gt[i].numpy().transpose(1, 2, 0)
                hazy_np = hazy[i].numpy().transpose(1, 2, 0)

                p = compute_psnr(out_np, gt_np)
                s = compute_ssim(out_np, gt_np)
                psnr_list.append(p)
                ssim_list.append(s)

                if len(samples) < 8:
                    fname = names[i] if isinstance(names, (list, tuple)) else names
                    extras = {}
                    if model_name == "DCPDN":
                        extras["transmission"] = t_map[i].cpu().numpy().squeeze()
                        extras["atmosphere"] = A_map[i].cpu().numpy().transpose(1, 2, 0)
                    samples.append((hazy_np, out_np, gt_np, fname, extras))

    return np.array(psnr_list), np.array(ssim_list), samples

## 4. Run Evaluation on RESIDE-SOTS

In [5]:
print("=" * 60)
print("Evaluating on RESIDE-SOTS")
print("=" * 60)

reside_dcp_psnr, reside_dcp_ssim, reside_dcp_samples = evaluate_dcp_on_loader(
    dcp, reside_loader, desc="DCP on RESIDE-SOTS")

reside_aod_psnr, reside_aod_ssim, reside_aod_samples = evaluate_model_on_loader(
    aodnet, reside_loader, "AOD-Net", device, desc="AOD-Net on RESIDE-SOTS")

reside_dcpdn_psnr, reside_dcpdn_ssim, reside_dcpdn_samples = evaluate_model_on_loader(
    dcpdn, reside_loader, "DCPDN", device, desc="DCPDN on RESIDE-SOTS")

print("\nRESIDE-SOTS evaluation complete.")

Evaluating on RESIDE-SOTS


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

## 5. Run Evaluation on O-HAZE

In [ ]:
print("=" * 60)
print("Evaluating on O-HAZE")
print("=" * 60)

ohaze_dcp_psnr, ohaze_dcp_ssim, ohaze_dcp_samples = evaluate_dcp_on_loader(
    dcp, ohaze_loader, desc="DCP on O-HAZE")

ohaze_aod_psnr, ohaze_aod_ssim, ohaze_aod_samples = evaluate_model_on_loader(
    aodnet, ohaze_loader, "AOD-Net", device, desc="AOD-Net on O-HAZE")

ohaze_dcpdn_psnr, ohaze_dcpdn_ssim, ohaze_dcpdn_samples = evaluate_model_on_loader(
    dcpdn, ohaze_loader, "DCPDN", device, desc="DCPDN on O-HAZE")

print("\nO-HAZE evaluation complete.")

## 6. Quantitative Results

In [ ]:
# Build results table
results = {
    "Method": ["DCP", "AOD-Net", "DCPDN"] * 2,
    "Dataset": ["RESIDE-SOTS"] * 3 + ["O-HAZE"] * 3,
    "PSNR (dB)": [
        reside_dcp_psnr.mean(), reside_aod_psnr.mean(), reside_dcpdn_psnr.mean(),
        ohaze_dcp_psnr.mean(), ohaze_aod_psnr.mean(), ohaze_dcpdn_psnr.mean(),
    ],
    "SSIM": [
        reside_dcp_ssim.mean(), reside_aod_ssim.mean(), reside_dcpdn_ssim.mean(),
        ohaze_dcp_ssim.mean(), ohaze_aod_ssim.mean(), ohaze_dcpdn_ssim.mean(),
    ],
    "PSNR Std": [
        reside_dcp_psnr.std(), reside_aod_psnr.std(), reside_dcpdn_psnr.std(),
        ohaze_dcp_psnr.std(), ohaze_aod_psnr.std(), ohaze_dcpdn_psnr.std(),
    ],
    "SSIM Std": [
        reside_dcp_ssim.std(), reside_aod_ssim.std(), reside_dcpdn_ssim.std(),
        ohaze_dcp_ssim.std(), ohaze_aod_ssim.std(), ohaze_dcpdn_ssim.std(),
    ],
}

df = pd.DataFrame(results)
df["PSNR (dB)"] = df["PSNR (dB)"].round(2)
df["SSIM"] = df["SSIM"].round(4)
df["PSNR Std"] = df["PSNR Std"].round(2)
df["SSIM Std"] = df["SSIM Std"].round(4)

print("\nQuantitative Results")
print("=" * 70)
df.style.highlight_max(
    subset=["PSNR (dB)", "SSIM"],
    props="font-weight: bold; color: green;"
)

## 7. Bar Chart Comparison

In [ ]:
methods = ["DCP", "AOD-Net", "DCPDN"]
x = np.arange(len(methods))
width = 0.3

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PSNR comparison
reside_psnr_means = [reside_dcp_psnr.mean(), reside_aod_psnr.mean(), reside_dcpdn_psnr.mean()]
ohaze_psnr_means = [ohaze_dcp_psnr.mean(), ohaze_aod_psnr.mean(), ohaze_dcpdn_psnr.mean()]
reside_psnr_stds = [reside_dcp_psnr.std(), reside_aod_psnr.std(), reside_dcpdn_psnr.std()]
ohaze_psnr_stds = [ohaze_dcp_psnr.std(), ohaze_aod_psnr.std(), ohaze_dcpdn_psnr.std()]

bars1 = axes[0].bar(x - width/2, reside_psnr_means, width, yerr=reside_psnr_stds,
                     label="RESIDE-SOTS", color="#4C72B0", capsize=4)
bars2 = axes[0].bar(x + width/2, ohaze_psnr_means, width, yerr=ohaze_psnr_stds,
                     label="O-HAZE", color="#DD8452", capsize=4)
axes[0].set_xlabel("Method")
axes[0].set_ylabel("PSNR (dB)")
axes[0].set_title("PSNR Comparison")
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)

# SSIM comparison
reside_ssim_means = [reside_dcp_ssim.mean(), reside_aod_ssim.mean(), reside_dcpdn_ssim.mean()]
ohaze_ssim_means = [ohaze_dcp_ssim.mean(), ohaze_aod_ssim.mean(), ohaze_dcpdn_ssim.mean()]
reside_ssim_stds = [reside_dcp_ssim.std(), reside_aod_ssim.std(), reside_dcpdn_ssim.std()]
ohaze_ssim_stds = [ohaze_dcp_ssim.std(), ohaze_aod_ssim.std(), ohaze_dcpdn_ssim.std()]

bars3 = axes[1].bar(x - width/2, reside_ssim_means, width, yerr=reside_ssim_stds,
                     label="RESIDE-SOTS", color="#4C72B0", capsize=4)
bars4 = axes[1].bar(x + width/2, ohaze_ssim_means, width, yerr=ohaze_ssim_stds,
                     label="O-HAZE", color="#DD8452", capsize=4)
axes[1].set_xlabel("Method")
axes[1].set_ylabel("SSIM")
axes[1].set_title("SSIM Comparison")
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

for bar in bars3:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars4:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle("Dehazing Performance: DCP vs AOD-Net vs DCPDN", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "bar_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()

## 8. PSNR Distribution (Box Plots)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RESIDE-SOTS box plot
bp1 = axes[0].boxplot(
    [reside_dcp_psnr, reside_aod_psnr, reside_dcpdn_psnr],
    labels=methods, patch_artist=True,
    boxprops=dict(facecolor='lightblue'),
    medianprops=dict(color='red', linewidth=2)
)
axes[0].set_title("RESIDE-SOTS - PSNR Distribution")
axes[0].set_ylabel("PSNR (dB)")
axes[0].grid(axis='y', alpha=0.3)

# O-HAZE box plot
bp2 = axes[1].boxplot(
    [ohaze_dcp_psnr, ohaze_aod_psnr, ohaze_dcpdn_psnr],
    labels=methods, patch_artist=True,
    boxprops=dict(facecolor='lightyellow'),
    medianprops=dict(color='red', linewidth=2)
)
axes[1].set_title("O-HAZE - PSNR Distribution")
axes[1].set_ylabel("PSNR (dB)")
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle("Per-Image PSNR Distribution", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "psnr_boxplot.png"), dpi=150, bbox_inches='tight')
plt.show()

## 9. Visual Comparison - RESIDE-SOTS

In [ ]:
def show_comparison(dcp_samples, aod_samples, dcpdn_samples, dataset_name, n_show=5):
    """Display side-by-side visual comparison."""
    n = min(n_show, len(dcp_samples), len(aod_samples), len(dcpdn_samples))
    fig, axes = plt.subplots(n, 5, figsize=(20, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    col_titles = ["Hazy Input", "Ground Truth", "DCP", "AOD-Net", "DCPDN"]

    for i in range(n):
        hazy, dcp_out, gt, name = dcp_samples[i]
        _, aod_out, _, _, _ = aod_samples[i]
        _, dcpdn_out, _, _, _ = dcpdn_samples[i]

        images = [hazy, gt, dcp_out, aod_out, dcpdn_out]
        for j, img in enumerate(images):
            axes[i, j].imshow(np.clip(img, 0, 1))
            axes[i, j].axis('off')
            if i == 0:
                axes[i, j].set_title(col_titles[j], fontsize=13, fontweight='bold')

    plt.suptitle(f"Visual Comparison - {dataset_name}", fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, f"visual_{dataset_name.replace('-', '_').lower()}.png"),
                dpi=150, bbox_inches='tight')
    plt.show()

show_comparison(reside_dcp_samples, reside_aod_samples, reside_dcpdn_samples,
                "RESIDE-SOTS", n_show=5)

## 10. Visual Comparison - O-HAZE

In [ ]:
show_comparison(ohaze_dcp_samples, ohaze_aod_samples, ohaze_dcpdn_samples,
                "O-HAZE", n_show=5)

## 11. DCPDN Intermediate Outputs

DCPDN estimates a **transmission map** and **atmospheric light map** as intermediate outputs. These are visualized below.

In [ ]:
def show_dcpdn_intermediates(dcpdn_samples, dataset_name, n_show=4):
    """Visualize DCPDN's intermediate transmission and atmospheric light maps."""
    n = min(n_show, len(dcpdn_samples))
    fig, axes = plt.subplots(n, 5, figsize=(22, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    col_titles = ["Hazy Input", "Transmission Map", "Atmospheric Light", "Dehazed", "Ground Truth"]

    for i in range(n):
        hazy, dehazed, gt, name, extras = dcpdn_samples[i]
        t_map = extras.get("transmission", np.zeros_like(hazy[:, :, 0]))
        a_map = extras.get("atmosphere", np.zeros_like(hazy))

        axes[i, 0].imshow(np.clip(hazy, 0, 1))
        axes[i, 0].axis('off')

        axes[i, 1].imshow(t_map, cmap='hot')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(np.clip(a_map, 0, 1))
        axes[i, 2].axis('off')

        axes[i, 3].imshow(np.clip(dehazed, 0, 1))
        axes[i, 3].axis('off')

        axes[i, 4].imshow(np.clip(gt, 0, 1))
        axes[i, 4].axis('off')

        if i == 0:
            for j, title in enumerate(col_titles):
                axes[i, j].set_title(title, fontsize=13, fontweight='bold')

    plt.suptitle(f"DCPDN Intermediate Outputs - {dataset_name}", fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, f"dcpdn_intermediates_{dataset_name.replace('-', '_').lower()}.png"),
                dpi=150, bbox_inches='tight')
    plt.show()

show_dcpdn_intermediates(reside_dcpdn_samples, "RESIDE-SOTS")
show_dcpdn_intermediates(ohaze_dcpdn_samples, "O-HAZE")

## 12. Per-Image PSNR Scatter: AOD-Net vs DCPDN

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RESIDE-SOTS scatter
min_len = min(len(reside_aod_psnr), len(reside_dcpdn_psnr))
lim_min = min(reside_aod_psnr[:min_len].min(), reside_dcpdn_psnr[:min_len].min()) - 1
lim_max = max(reside_aod_psnr[:min_len].max(), reside_dcpdn_psnr[:min_len].max()) + 1

axes[0].scatter(reside_aod_psnr[:min_len], reside_dcpdn_psnr[:min_len],
                alpha=0.4, s=15, color='#4C72B0')
axes[0].plot([lim_min, lim_max], [lim_min, lim_max], 'r--', alpha=0.7, label='y = x')
axes[0].set_xlabel("AOD-Net PSNR (dB)")
axes[0].set_ylabel("DCPDN PSNR (dB)")
axes[0].set_title("RESIDE-SOTS")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_xlim(lim_min, lim_max)
axes[0].set_ylim(lim_min, lim_max)
axes[0].set_aspect('equal')

# O-HAZE scatter
min_len2 = min(len(ohaze_aod_psnr), len(ohaze_dcpdn_psnr))
lim_min2 = min(ohaze_aod_psnr[:min_len2].min(), ohaze_dcpdn_psnr[:min_len2].min()) - 1
lim_max2 = max(ohaze_aod_psnr[:min_len2].max(), ohaze_dcpdn_psnr[:min_len2].max()) + 1

axes[1].scatter(ohaze_aod_psnr[:min_len2], ohaze_dcpdn_psnr[:min_len2],
                alpha=0.6, s=30, color='#DD8452')
axes[1].plot([lim_min2, lim_max2], [lim_min2, lim_max2], 'r--', alpha=0.7, label='y = x')
axes[1].set_xlabel("AOD-Net PSNR (dB)")
axes[1].set_ylabel("DCPDN PSNR (dB)")
axes[1].set_title("O-HAZE")
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_xlim(lim_min2, lim_max2)
axes[1].set_ylim(lim_min2, lim_max2)
axes[1].set_aspect('equal')

plt.suptitle("Per-Image PSNR: AOD-Net vs DCPDN", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "scatter_aod_vs_dcpdn.png"), dpi=150, bbox_inches='tight')
plt.show()

# Stats
dcpdn_wins_reside = (reside_dcpdn_psnr[:min_len] > reside_aod_psnr[:min_len]).sum()
dcpdn_wins_ohaze = (ohaze_dcpdn_psnr[:min_len2] > ohaze_aod_psnr[:min_len2]).sum()
print(f"RESIDE-SOTS: DCPDN wins on {dcpdn_wins_reside}/{min_len} images ({100*dcpdn_wins_reside/min_len:.1f}%)")
print(f"O-HAZE: DCPDN wins on {dcpdn_wins_ohaze}/{min_len2} images ({100*dcpdn_wins_ohaze/min_len2:.1f}%)")

## 13. Summary Table

In [ ]:
# Pivot table for cleaner display
summary = df.pivot_table(index='Method', columns='Dataset', values=['PSNR (dB)', 'SSIM'])
summary = summary.reindex(["DCP", "AOD-Net", "DCPDN"])
print("\nFinal Summary")
print("=" * 60)
summary

## 14. Model Complexity

Compare the number of trainable parameters in each deep learning model.

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

aod_params = count_params(aodnet)
dcpdn_params = count_params(dcpdn)

param_df = pd.DataFrame({
    "Model": ["AOD-Net", "DCPDN"],
    "Parameters": [aod_params, dcpdn_params],
    "Size (approx)": [f"{aod_params * 4 / 1024:.1f} KB", f"{dcpdn_params * 4 / 1024 / 1024:.1f} MB"],
})
print("Model Complexity")
print("=" * 40)
param_df

In [ ]:
# Parameter comparison bar chart
fig, ax = plt.subplots(figsize=(6, 4))
colors = ["#4C72B0", "#C44E52"]
bars = ax.bar(["AOD-Net", "DCPDN"], [aod_params, dcpdn_params], color=colors)
ax.set_ylabel("Number of Parameters")
ax.set_title("Model Complexity Comparison")
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, [aod_params, dcpdn_params]):
    if val > 1e6:
        label = f"{val/1e6:.2f}M"
    else:
        label = f"{val/1e3:.1f}K"
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            label, ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "model_complexity.png"), dpi=150, bbox_inches='tight')
plt.show()

## 15. Conclusions

| Aspect | DCP | AOD-Net | DCPDN |
|--------|-----|---------|-------|
| **Type** | Traditional (no training) | Lightweight CNN | Dense pyramid + GAN |
| **Parameters** | 0 | ~2K | ~4M |
| **Key Idea** | Dark channel + guided filter | K(x) reformulation | Transmission + atm. light estimation |
| **Strengths** | No data needed, fast | Small model, easy to train | Physics-based, rich intermediates |
| **Weaknesses** | Fails on bright scenes | Limited capacity | Slow training, large model |